# 180. Consecutive Numbers

**Difficulty:** Medium &nbsp;|&nbsp; **Topics:** database, self-join, window-function
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/consecutive-numbers/)

```
Table: Logs
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| id          | int     |
| num         | varchar |
+-------------+---------+
id is the primary key and an autoincrement column starting from 1.
```

Find all numbers that appear **at least three times consecutively**.

Return the result table in **any order**. The result column must be called
`ConsecutiveNums`.

---

### Example

```
Logs:                            Output:
+----+-----+                     +-----------------+
| id | num |                     | ConsecutiveNums |
+----+-----+                     +-----------------+
| 1  | 1   |                     | 1               |
| 2  | 1   |                     +-----------------+
| 3  | 1   |
| 4  | 2   |
| 5  | 1   |
| 6  | 2   |
| 7  | 2   |
+----+-----+
```

`1` is the only number that appears three times consecutively (ids 1, 2, 3). The `2`s
at ids 6 and 7 are only two in a row, and the `1` at id 5 is on its own.

---

"Consecutive" is a word about **order**, and a table has no order - only the order you
give it with a column. This is the first problem here where the answer depends on
comparing a row to its *neighbours*, and there are two completely different ways to
reach a neighbour in SQL.

## Before you write anything

**1.** Write down what "three consecutive rows have the same num" means as a condition
on **three rows at once**. SQL compares columns within one row, so - as in #181 - you
must first build a result where all three are side by side. Name the two ways to do
that. (One you already know from #181; the other is newer and is called a *window
function*.)

**2.** **The self join.** `Logs` three times, aliased `a`, `b`, `c`, joined on
`b.id = a.id + 1` and `c.id = a.id + 2`, then filtered on all three `num`s being equal.
Write it. Now say precisely what it is assuming about `id` - and go back and read the
schema line about autoincrement. This query is only correct because of that guarantee.

**3.** **The window function.** `LAG(num, 1) OVER (ORDER BY id)` gives you the `num` of
the previous row; `LAG(num, 2)` the one before that. Write the version that compares
`num` against both. What does `LAG` return for the first two rows, and does that break
your comparison or handle itself? (Think about what `1 = NULL` evaluates to - the same
rule from #181 and #183, helping you for the third time.)

**4.** The `ORDER BY id` inside `OVER (...)` is doing something the self join could not:
it defines "previous" by **rank in an ordering**, not by arithmetic on the id. So what
happens to each version if the ids are `1, 2, 5, 6, 7` - a real table after some rows
were deleted? Predict both, then look at the test case named for it below. One of the
two routes fails it, on purpose.

**5.** The output must be **distinct**. If a number appears five times in a row, how
many rows does your join produce for it, and how many should be reported? Which keyword
fixes it?

**6.** Four in a row - does that count? Read the statement's "at least three" again, and
check that your query says yes without a special case.

## Two routes

**A - the three-way self join** *(write this first; it is the classic answer)*

```sql
SELECT DISTINCT a.num AS ConsecutiveNums
FROM Logs a
JOIN Logs b ON b.id = a.id + 1
JOIN Logs c ON c.id = a.id + 2
WHERE a.num = b.num AND b.num = c.num
```

`a` is the first of the three, `b` the next, `c` the one after. `DISTINCT` collapses a
run of five into a single reported row (question 5). It is #181's self join with one
more copy of the table, and it is what most people write.

Its correctness rests entirely on `id` being a gapless autoincrement, which LeetCode
guarantees and real tables do not.

**B - `LAG`, a window function**

```sql
SELECT DISTINCT num AS ConsecutiveNums
FROM (
    SELECT num,
           LAG(num, 1) OVER (ORDER BY id) AS prev1,
           LAG(num, 2) OVER (ORDER BY id) AS prev2
    FROM Logs
) t
WHERE num = prev1 AND num = prev2
```

`LAG(num, k) OVER (ORDER BY id)` means "the `num` from `k` rows earlier, when the rows
are sorted by `id`" - by **position**, not by arithmetic. For the first two rows `LAG`
returns `NULL`, and `num = NULL` is never true, so they exclude themselves.

This version does not care whether ids are gapless, which is why it still works on the
dataset with gaps and route A does not. It is also how you would write it at work,
where an `id` column with holes in it is the normal case.

> **A table has no order until you give it one.** The self join fakes an ordering out of
> arithmetic on the ids and inherits every assumption that comes with it. `OVER (ORDER
> BY id)` states the ordering explicitly, which is both clearer and more robust. Write A
> because you will be asked for it; reach for B when the data is real.

In [ ]:
SOLUTION = '''
'''

### The test harness

Every notebook in this folder runs your SQL for real, against a fresh **SQLite**
database built from scratch for each test case. Nothing is mocked and nothing is
pattern-matched - if your query runs and returns the right rows, it passes.

`check(name, data, expected)` creates the tables, inserts that case's rows, executes
whatever string is in `SOLUTION`, and compares. It checks two things: the **rows**
(as a set - row order does not matter unless the problem says it does) and the
**column names**, because a query that returns the right numbers under the wrong
headings is not the answer the question asked for.

On failure it prints your rows next to the expected ones and names which rows are
missing and which should not be there.

`show(name, data, query)` is there for you: run *any* query against any dataset and
print it. Use it to look at intermediate results while you are working - especially
to run the deliberately-wrong version of your query and watch what it does.

> **SQLite here, MySQL on LeetCode.** They agree on everything these problems need -
> joins, `GROUP BY`/`HAVING`, subqueries, `LIMIT`/`OFFSET`, `COALESCE`, and window
> functions like `DENSE_RANK`. Where a problem needs something MySQL does differently,
> the notebook says so in the routes section. Write standard SQL and both will take it.

Run this cell; don't edit it.

In [ ]:
import sqlite3

SCHEMA = """CREATE TABLE Logs (id INTEGER, num INTEGER);"""

EXPECTED_COLUMNS = ['ConsecutiveNums']
ORDERED = False


def _norm(rows):
    return rows if ORDERED else sorted(rows, key=lambda r: tuple((v is None, str(v)) for v in r))


def check(name, data_sql, expected):
    """Build a fresh in-memory database, run SOLUTION against it, compare."""
    con = sqlite3.connect(":memory:")
    try:
        con.executescript(SCHEMA)
        if data_sql.strip():
            con.executescript(data_sql)
    except sqlite3.Error as e:
        print(f"FAIL {name}")
        print(f"       the harness could not build the tables: {e}")
        return False

    if not SOLUTION.strip():
        print(f"FAIL {name}")
        print("       SOLUTION is empty - write your query in the cell above")
        return False

    try:
        cur = con.execute(SOLUTION)
        got = [tuple(r) for r in cur.fetchall()]
        cols = [d[0] for d in cur.description] if cur.description else []
    except sqlite3.Error as e:
        print(f"FAIL {name}")
        print(f"       your query raised {type(e).__name__}: {e}")
        return False

    cols_ok = [c.lower() for c in cols] == [c.lower() for c in EXPECTED_COLUMNS]
    rows_ok = _norm(got) == _norm(expected)

    if cols_ok and rows_ok:
        print(f"OK   {name}")
        return True

    print(f"FAIL {name}")
    if not cols_ok:
        print(f"       column names  {cols}")
        print(f"       should be     {EXPECTED_COLUMNS}")
    if not rows_ok:
        missing = [r for r in expected if r not in got]
        extra = [r for r in got if r not in expected]
        print(f"       you returned {len(got)} row(s), expected {len(expected)}"
              + ("   (row order matters here)" if ORDERED else "   (row order does not matter)"))
        for r in got[:6]:
            print(f"         got       {r}")
        for r in expected[:6]:
            print(f"         expected  {r}")
        if missing:
            print(f"       rows you are MISSING: {missing[:4]}")
        if extra:
            print(f"       rows you should NOT have: {extra[:4]}")
    return False


def show(name, data_sql, query):
    """Run any query against a dataset and print it - for exploring, not for grading."""
    con = sqlite3.connect(":memory:")
    con.executescript(SCHEMA)
    if data_sql.strip():
        con.executescript(data_sql)
    cur = con.execute(query)
    cols = [d[0] for d in cur.description]
    rows = cur.fetchall()
    print(f"-- {name}")
    print("   " + " | ".join(str(c) for c in cols))
    for r in rows:
        print("   " + " | ".join("NULL" if v is None else str(v) for v in r))
    if not rows:
        print("   (no rows)")

In [ ]:
# tests
check("the LeetCode example", '''
INSERT INTO Logs VALUES (1,1),(2,1),(3,1),(4,2),(5,1),(6,2),(7,2);
''', [(1,)])

check("nothing repeats", '''
INSERT INTO Logs VALUES (1,1),(2,2),(3,3),(4,4);
''', [])

check("exactly two in a row is not enough", '''
INSERT INTO Logs VALUES (1,5),(2,5),(3,9);
''', [])

check("exactly three in a row", '''
INSERT INTO Logs VALUES (1,5),(2,5),(3,5);
''', [(5,)])

check("question 5+6: five in a row is reported ONCE", '''
INSERT INTO Logs VALUES (1,7),(2,7),(3,7),(4,7),(5,7);
''', [(7,)])

check("two different numbers both qualify", '''
INSERT INTO Logs VALUES (1,1),(2,1),(3,1),(4,2),(5,2),(6,2);
''', [(1,),(2,)])

check("the same number qualifies twice in separate runs", '''
INSERT INTO Logs VALUES (1,3),(2,3),(3,3),(4,9),(5,3),(6,3),(7,3);
''', [(3,)])

check("a run that is broken by one row", '''
INSERT INTO Logs VALUES (1,4),(2,4),(3,9),(4,4),(5,4);
''', [])

check("fewer than three rows in the table", '''
INSERT INTO Logs VALUES (1,1),(2,1);
''', [])

check("an empty table", '', [])

check("negative numbers and zero", '''
INSERT INTO Logs VALUES (1,0),(2,0),(3,0),(4,-1),(5,-1);
''', [(0,)])

check("*** question 4: ids with GAPS - route A fails this, route B does not ***", '''
INSERT INTO Logs VALUES (1,8),(2,8),(5,8),(6,3),(7,3),(8,3);
''', [(8,),(3,)])

## After it passes

- **Run both routes against the gappy dataset** with `show` and put the results side by
  side. Route A misses the run of `8`s because `id + 1` walked into a hole. Write one
  sentence about what route A was really testing - it was never "is the next row", it
  was "is there a row numbered one higher", and those stopped being the same thing the
  moment somebody deleted a row.
- **Delete a row from a real-looking table** and watch it happen: take the LeetCode
  dataset, `DELETE FROM Logs WHERE id = 2`, and re-run route A. The bug arrives without
  anybody touching the query.
- **Then generalise to N.** "At least four consecutively" means one more join for route
  A and one more `LAG` for route B. Which of the two would you rather change for
  `N = 10`? That question is the honest argument for window functions, and it is worth
  more than any performance claim.
- **The grown-up version.** Report the *runs* themselves - number, start id, length -
  rather than just which numbers had one. The standard technique is called *gaps and
  islands*, and it is `ROW_NUMBER()` minus a second `ROW_NUMBER()` partitioned by `num`.
  Look it up after you have finished this; it is the single most useful SQL pattern
  that is not taught in tutorials.
- Siblings: #181 Employees Earning More Than Their Managers (the self join, two copies),
  #603 Consecutive Available Seats, #1204 Last Person to Fit in the Bus (running totals
  with a window), #178 Rank Scores.